# 📊 Exploratory Data Analysis (EDA)

**Dự án:** Phân loại giới tính qua giọng nói (Gender Voice Classification)

Mục tiêu của notebook này:
1. Khám phá cấu trúc và phân phối của dataset (số lượng mẫu).
2. Phân tích tình trạng mất cân bằng dữ liệu (Class Imbalance).
3. Khám phá phân phối về độ dài (duration) và sample rate của các file audio.
4. Trực quan hóa tín hiệu âm thanh (Waveform) cho Nam vs Nữ.
5. Trực quan hóa đặc trưng âm thanh (MFCC Spectrogram) cho Nam vs Nữ.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Dataset Overview & Class Distribution
Đầu tiên, chúng ta sẽ đếm số lượng file trong từng thư mục để xem phân phối của 2 lớp (Nam và Nữ).

In [ ]:
dataset_dir = '../dataset'
male_dir = os.path.join(dataset_dir, 'male')
female_dir = os.path.join(dataset_dir, 'female')

def get_wav_files(directory):
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith('.wav')]

male_files = get_wav_files(male_dir)
female_files = get_wav_files(female_dir)

num_male = len(male_files)
num_female = len(female_files)
total_files = num_male + num_female

print(f"Tổng số files audio: {total_files}")
print(f"  - Male (Nam)  : {num_male} files ({num_male/total_files*100:.2f}%)")
print(f"  - Female (Nữ) : {num_female} files ({num_female/total_files*100:.2f}%)")

In [ ]:
# Plot Class Distribution
plt.figure(figsize=(8, 6))
sns.barplot(x=['Male', 'Female'], y=[num_male, num_female], palette=['#3498db', '#e74c3c'])
plt.title('Class Distribution: Male vs Female', fontsize=16, fontweight='bold')
plt.ylabel('Number of Audio Files', fontsize=12)
plt.xlabel('Gender', fontsize=12)

for i, v in enumerate([num_male, num_female]):
    plt.text(i, v + 100, str(v), ha='center', fontsize=12, fontweight='bold')

plt.show()

> **Nhận xét:** Dataset có sự chênh lệch (Imbalance). Số lượng file Nam nhiều hơn Nữ đáng kể (8420 vs 5776). Trong quá trình huấn luyện, chúng ta sẽ cần sử dụng các kỹ thuật như Class Weights hoặc Augmentation để xử lý vấn đề này.

## 2. Audio Duration and Sample Rate Analysis
Lấy mẫu ngẫu nhiên khoảng 1000 file (để tăng tốc độ tính toán) nhằm kiểm tra độ dài trung bình và sample rate của các file âm thanh.

In [ ]:
import random

# Randomly sample 1000 files to analyze duration and sr
sample_size = min(1000, total_files)
all_files = male_files + female_files
sampled_files = random.sample(all_files, sample_size)

durations = []
sample_rates = []

print(f"Analyzing {sample_size} randomly sampled files...")
for f in tqdm(sampled_files):
    # Using librosa to get duration is faster if we don't load the full audio array
    duration = librosa.get_duration(path=f)
    sr = librosa.get_samplerate(f)
    durations.append(duration)
    sample_rates.append(sr)

In [ ]:
# Plot Duration Distribution
plt.figure(figsize=(10, 6))
sns.histplot(durations, bins=50, kde=True, color='purple')
plt.title('Distribution of Audio Durations', fontsize=16, fontweight='bold')
plt.xlabel('Duration (seconds)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(np.mean(durations), color='red', linestyle='dashed', linewidth=2, label=f'Mean: {np.mean(durations):.2f}s')
plt.axvline(np.median(durations), color='green', linestyle='dashed', linewidth=2, label=f'Median: {np.median(durations):.2f}s')
plt.legend()
plt.show()

# Print SR distribution
sr_counts = pd.Series(sample_rates).value_counts()
print("Sample Rates Distribution:")
print(sr_counts)

> **Nhận xét:** 
>- Đa số các file âm thanh có độ dài từ 1.5 đến 4 giây. Việc cắt/padding các file về một độ dài cố định (ví dụ: 3 giây) là một quyết định hợp lý cho quá trình tiền xử lý.
>- Chúng ta cũng kiểm tra Sample Rate để đảm bảo dữ liệu đồng nhất.

## 3. Waveform and Spectrogram Visualization
Trực quan hóa một file mẫu của Nam và một file mẫu của Nữ để so sánh đặc trưng.

In [ ]:
def plot_waveform(file_path, title, color):
    y, sr = librosa.load(file_path, sr=16000)
    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(y, sr=sr, color=color)
    plt.title(f'Waveform - {title}', fontsize=14, fontweight='bold')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.show()
    return y, sr

# Select random samples
sample_male = random.choice(male_files)
sample_female = random.choice(female_files)

print("Sample Male:", os.path.basename(sample_male))
y_male, sr_male = plot_waveform(sample_male, 'Male', color='#3498db')

print("Sample Female:", os.path.basename(sample_female))
y_female, sr_female = plot_waveform(sample_female, 'Female', color='#e74c3c')

### MFCC (Mel-Frequency Cepstral Coefficients)
MFCC là đặc trưng quan trọng nhất được sử dụng trong bài toán nhận dạng giọng nói.

In [ ]:
def plot_mfcc(y, sr, title):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(mfccs, x_axis='time', sr=sr, cmap='viridis')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'MFCC - {title}', fontsize=14, fontweight='bold')
    plt.ylabel('MFCC Coefficients')
    plt.xlabel('Time (s)')
    plt.show()

plot_mfcc(y_male, sr_male, 'Male')
plot_mfcc(y_female, sr_female, 'Female')

### Tổng kết EDA:
1. **Tổng số lượng dữ liệu:** ~14.2K files (Male: 8420, Female: 5776).
2. **Độ dài audio:** Đa phần tập trung từ 1.5s - 4.0s. Độ dài 3.0s là lý tưởng để cắt gọt (padding/trimming).
3. **Đặc trưng âm thanh:** MFCC thể hiện rõ nét sự khác biệt về cao độ (pitch) và âm sắc giữa giọng nam (thường trầm hơn) và giọng nữ (thường thanh hơn).

➡️ **Hướng xử lý dữ liệu cho Model:**
- Khử nhiễu, chuẩn hóa.
- Trích xuất 20 hệ số MFCC + Delta + Delta² (60 tính năng).
- Resample tất cả về 16kHz.
- Cắt/pad độ dài chuẩn 3.0s.
- Data Augmentation (Thêm nhiễu, Dịch chuyển thời gian) trên tập Train để giải quyết mất cân bằng lớp.